In [ ]:
import pandas as pd
import openslide
from PIL import Image
from os.path import join as opj
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
all_meta = pd.read_csv("/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/master_spreadsheet/matched_su_mxa.csv")

In [ ]:
frozens = all_meta[all_meta["Block"].str.contains("FS")]
notfroz = all_meta[~all_meta["Block"].str.contains("FS")]

In [ ]:
images = []

In [ ]:
svs_root = "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/svs"

for i, s in frozens.sample(500).iterrows():
    if not type(s["path"]) == str:
        continue
    slide = openslide.OpenSlide(opj(svs_root, s["path"]))
    macro1 = np.array(slide.associated_images["macro"])[:100,-100:,:]
    macro2 = np.array(slide.associated_images["macro"])[-100:,-100:,:]
    images.append(np.stack((macro1, macro2)))

In [ ]:
#np.save("plusi.npy", np.stack(images))

In [ ]:
import cv2
import einops
from sklearn.manifold import TSNE

In [ ]:
smol_im = [1-cv2.resize(cv2.cvtColor(i, cv2.COLOR_RGB2GRAY), (28,28)) for i in images]
features = einops.rearrange(np.stack(smol_im), "n h w -> n (h w)")
tsne = TSNE(n_components=2)
xy = tsne.fit_transform(features)

In [ ]:
import altair as alt
import base64
from io import BytesIO

In [ ]:
def encode_image(img):
    buffered = BytesIO()
    Image.fromarray(img).save(buffered, format="PNG")
    img_str =  base64.b64encode(buffered.getvalue()).decode()
    return f"data:image/png;base64,{img_str}"

# Convert images to base64 for tooltip display
image_base64 = [encode_image(i) for i in images]

# Create DataFrame for visualization
df = pd.DataFrame({
    'x': xy[:, 0],
    'y': xy[:, 1],
    'image': image_base64
})

# Create Altair plot
chart = alt.Chart(df).mark_circle(size=60).encode(
    x='x:Q',
    y='y:Q',
    tooltip=["image"]
).interactive()


In [ ]:
chart
